DIY Measles Tracking Dashboard Kit. Adapted from the original [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash) by Fabrizio Smeraldi. Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/).

# DIY Measles Tracking Dashboard

This is a template for your DIY Measles Tracking Dashboard, where you can analyse and visualise UKHSA measles data. The dashboard is displayed using [voila](https://voila.readthedocs.io/en/stable/index.html), which converts notebooks to standalone dashboards.

To render this notebook with Voila in Binder, use your own repository URL with `?urlpath=%2Fvoila%2Frender%2FDashboard.ipynb`.


In [ ]:
from IPython.display import clear_output
import ipywidgets as wdg
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import time
import json

In [ ]:
%matplotlib inline
# make figures larger
plt.rcParams['figure.dpi'] = 100

## Load initial data from disk

Default dataset: UKHSA Measles weekly cases (England). Use the Area selector + Fetch Data to switch to a UKHSA region from the same measles dataset.

In [ ]:
# Configure the local canned dataset and how to parse it.
# This file is used on first load; the refresh button fetches live UKHSA Measles data.
DATASET_CONFIG = {
    "json_path": "measles_timeseries.json",
    "records_path": ["data"],
    "date_field": "date",
    "value_fields": ["cases"],
    "rename": {},
    "fillna_value": None,
}

def _get_nested_value(payload, path):
    current = payload
    for key in path:
        current = current[key]
    return current

with open(DATASET_CONFIG["json_path"], "rt") as infile:
    jsondata = json.load(infile)


## Wrangle the data

`wrangle_data()` converts the configured JSON records into a clean `DataFrame` indexed by date.

In [ ]:
def wrangle_data(rawdata):
    """Return a time-indexed dataframe from the configured JSON structure."""
    records = _get_nested_value(rawdata, DATASET_CONFIG["records_path"])
    frame = pd.DataFrame.from_records(records)

    date_field = DATASET_CONFIG["date_field"]
    value_fields = DATASET_CONFIG["value_fields"]
    required_columns = [date_field] + value_fields
    missing_columns = [col for col in required_columns if col not in frame.columns]
    if missing_columns:
        raise KeyError(f"Missing columns in dataset: {missing_columns}")

    df = frame[required_columns].copy()
    rename_map = DATASET_CONFIG.get("rename", {})
    df.rename(columns=rename_map, inplace=True)

    df[date_field] = pd.to_datetime(df[date_field], errors="coerce")
    df.dropna(subset=[date_field], inplace=True)
    df.set_index(date_field, inplace=True)

    numeric_cols = [rename_map.get(col, col) for col in value_fields]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df.sort_index(inplace=True)

    fillna_value = DATASET_CONFIG.get("fillna_value")
    if fillna_value is not None:
        df.fillna(fillna_value, inplace=True)

    return df

df = wrangle_data(jsondata)


## Download current data

Choose an area (England or a UKHSA region), then click the button to refresh live UKHSA Measles data for that area.

In [ ]:
from urllib.parse import quote

API_BASE = "https://api.ukhsa-dashboard.data.gov.uk"

API_STRUCTURE = {
    "theme": "infectious_disease",
    "sub_theme": "vaccine_preventable",
    "topic": "Measles",
}

AREA_OPTIONS = [
    "England",
    "East Midlands",
    "East of England",
    "London",
    "North East",
    "North West",
    "South East",
    "South West",
    "West Midlands",
    "Yorkshire and Humber",
]

API_METRICS = {
    "cases": "measles_cases_casesByOnsetWeek",
}

API_PARAMS = {
    "format": "json",
    "age": "all",
    "sex": "all",
    "stratum": "default",
    "page_size": 500,
}

def _resolve_geography(area):
    if area == "England":
        return "Nation", "England"
    return "UKHSA Region", area

def _metric_endpoint(metric_name, geography_type, geography):
    return (
        f"{API_BASE}/themes/{quote(API_STRUCTURE['theme'])}"
        f"/sub_themes/{quote(API_STRUCTURE['sub_theme'])}"
        f"/topics/{quote(API_STRUCTURE['topic'])}"
        f"/geography_types/{quote(geography_type)}"
        f"/geographies/{quote(geography)}"
        f"/metrics/{quote(metric_name)}"
    )

def _fetch_all_results(url, params):
    rows = []
    next_url = url
    first_request = True

    while next_url:
        request_params = params if first_request else None
        response = requests.get(next_url, params=request_params, timeout=30)
        response.raise_for_status()
        payload = response.json()
        rows.extend(payload.get("results", []))

        next_url = payload.get("next")
        first_request = False

    return rows

def access_api(area):
    """Fetch live UKHSA Measles data for the selected area."""
    if not API_METRICS:
        return jsondata

    geography_type, geography = _resolve_geography(area)

    merged = {}
    for column, metric in API_METRICS.items():
        endpoint = _metric_endpoint(metric, geography_type, geography)
        for row in _fetch_all_results(endpoint, API_PARAMS):
            date = row.get("date")
            value = row.get("metric_value")
            if not date:
                continue

            record = merged.setdefault(date, {"date": date})
            record[column] = value

    data = [merged[date] for date in sorted(merged)]
    return {"data": data}


In [ ]:
status = wdg.HTML(value="")

area = wdg.Dropdown(
    options=AREA_OPTIONS,
    value="England",
    description="Area:",
    disabled=False,
)

def api_button_callback(button):
    """Refresh data and update the dataframe used for plotting."""
    global df
    global jsondata

    selected_area = area.value

    try:
        apidata = access_api(selected_area)
        df = wrangle_data(apidata)
        jsondata = apidata

        metrics = list(df.columns)

        if "series" in globals():
            series.options = metrics
            series.value = tuple(metrics[: min(3, len(metrics))])

        if "weeks" in globals():
            weeks.max = max(4, len(df))
            if weeks.value > weeks.max:
                weeks.value = weeks.max

        if "change_metric" in globals():
            change_metric.options = metrics
            if metrics and change_metric.value not in metrics:
                change_metric.value = metrics[0]

        if "change_weeks" in globals():
            change_weeks.max = max(4, len(df) - 1)
            if change_weeks.value > change_weeks.max:
                change_weeks.value = change_weeks.max

        if "refresh_graph" in globals():
            refresh_graph()

        if "refresh_weekly_change" in globals():
            refresh_weekly_change()

        apibutton.icon = "check"
        apibutton.button_style = "success"
        status.value = f"<span style='color: #2e7d32;'>Data refreshed successfully for {selected_area}.</span>"
    except Exception as err:
        apibutton.icon = "times"
        apibutton.button_style = "warning"
        status.value = f"<span style='color: #b71c1c;'>Refresh failed for {selected_area}: {err}</span>"


apibutton = wdg.Button(
    description="Fetch data",
    disabled=False,
    button_style="",
    tooltip="Download current data",
    icon="download",
)

apibutton.on_click(api_button_callback)
display(wdg.HBox([area, apibutton]))
display(status)


## Graphs and Analysis

Use the controls below to explore four views: weekly bars with a rolling average, week-over-week changes, age distribution, and regional ranking.

In [ ]:
def _clean_timeseries_frame(frame):
    out = frame.copy()
    out.index = pd.to_datetime(out.index, errors="coerce")
    out = out[~out.index.isna()]
    out.sort_index(inplace=True)
    return out


def plot_timeseries(gcols, gscale, nweeks, rolling_window):
    if len(gcols) == 0:
        print("Select one or more metrics to plot.")
        return

    plotdf = _clean_timeseries_frame(df[list(gcols)]).tail(nweeks)
    if plotdf.empty:
        print("No valid dated rows to plot.")
        return

    labels = plotdf.index.strftime("%Y-%m-%d")
    bar_df = plotdf.copy()
    bar_df.index = labels
    logscale = gscale == "log"

    ax = bar_df.plot(kind="bar", logy=logscale, figsize=(12, 4), alpha=0.85)
    x_positions = np.arange(len(plotdf))
    for col in gcols:
        rolling = plotdf[col].rolling(window=rolling_window, min_periods=1).mean()
        ax.plot(x_positions, rolling.values, marker="o", linewidth=2, label=f"{col} ({rolling_window}w avg)")

    ax.set_title("Weekly cases with rolling average")
    ax.set_xlabel("Week beginning")
    ax.set_ylabel("Cases")
    ax.tick_params(axis="x", labelrotation=75)
    ax.legend(loc="upper left", ncol=2)
    plt.tight_layout()
    plt.show()


def plot_weekly_change(metric, nweeks):
    if metric not in df.columns:
        print("Metric not available.")
        return

    change_df = _clean_timeseries_frame(df[[metric]]).tail(nweeks + 1)
    if len(change_df) < 2:
        print("Not enough data points to calculate week-over-week change.")
        return

    deltas = change_df[metric].diff().dropna()
    labels = deltas.index.strftime("%Y-%m-%d")
    colors = np.where(deltas >= 0, "#2ca02c", "#d62728")

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.bar(labels, deltas.values, color=colors)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"Week-over-week change for {metric}")
    ax.set_xlabel("Week beginning")
    ax.set_ylabel("Change in cases")
    ax.tick_params(axis="x", labelrotation=75)
    plt.tight_layout()
    plt.show()


def _load_age_distribution(path="agedistribution.json"):
    try:
        with open(path, "rt") as infile:
            payload = json.load(infile)
    except Exception as err:
        return pd.DataFrame(), {"error": str(err)}

    rows = payload.get("data", [])
    frame = pd.DataFrame.from_records(rows)
    if frame.empty:
        return frame, payload.get("source", {})

    required = ["age_group", "percent_cases"]
    missing = [col for col in required if col not in frame.columns]
    if missing:
        return pd.DataFrame(), {"error": f"Missing age distribution columns: {missing}"}

    frame["percent_cases"] = pd.to_numeric(frame["percent_cases"], errors="coerce")
    frame.dropna(subset=["percent_cases"], inplace=True)
    return frame, payload.get("source", {})


age_distribution_df, age_distribution_meta = _load_age_distribution()


def plot_age_distribution():
    if age_distribution_df.empty:
        error_msg = age_distribution_meta.get("error", "No age distribution data available.")
        print(error_msg)
        return

    plotdf = age_distribution_df.sort_values("percent_cases", ascending=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(plotdf["age_group"], plotdf["percent_cases"], color="#4c78a8")
    max_val = plotdf["percent_cases"].max()
    for idx, val in enumerate(plotdf["percent_cases"]):
        ax.text(val + max(0.2, max_val * 0.01), idx, f"{val:.1f}%", va="center")
    ax.set_title("Cumulative measles cases by age group (percent)")
    ax.set_xlabel("Percent of cases")
    ax.set_ylabel("Age group")
    plt.tight_layout()
    plt.show()

    reporting_period = age_distribution_meta.get("reporting_period")
    if reporting_period:
        print(reporting_period)


regional_cases_df = pd.DataFrame()


def _build_regional_cases_table():
    areas = ["England"] + [region for region in AREA_OPTIONS if region != "England"]
    area_frames = []
    errors = []

    for region in areas:
        try:
            region_data = access_api(region)
            region_df = wrangle_data(region_data)
            region_df = _clean_timeseries_frame(region_df[["cases"]])
            region_df.rename(columns={"cases": region}, inplace=True)
            area_frames.append(region_df)
        except Exception as err:
            errors.append(f"{region}: {err}")

    if area_frames:
        merged = pd.concat(area_frames, axis=1)
        merged.sort_index(inplace=True)
    else:
        merged = pd.DataFrame()

    return merged, errors


def _set_ranking_week_options():
    if regional_cases_df.empty:
        ranking_week.options = []
        ranking_week.disabled = True
        return

    week_labels = [dt.strftime("%Y-%m-%d") for dt in regional_cases_df.index]
    week_labels.sort(reverse=True)
    ranking_week.options = week_labels
    ranking_week.value = week_labels[0]
    ranking_week.disabled = False


def fetch_regional_ranking_data(_button):
    global regional_cases_df

    ranking_status.value = "<span style='color: #1565c0;'>Fetching regional data...</span>"
    regional_button.disabled = True
    regional_button.icon = "hourglass"
    regional_button.button_style = ""

    try:
        regional_cases_df, fetch_errors = _build_regional_cases_table()
        if regional_cases_df.empty:
            raise ValueError("No regional data available.")

        _set_ranking_week_options()
        ranking_status.value = (
            f"<span style='color: #2e7d32;'>Regional ranking data refreshed for {len(regional_cases_df.columns)} areas.</span>"
        )

        if fetch_errors:
            joined_errors = "; ".join(fetch_errors[:3])
            ranking_status.value += (
                f"<br><span style='color: #b26a00;'>Some areas failed: {joined_errors}</span>"
            )

        regional_button.icon = "check"
        regional_button.button_style = "success"
    except Exception as err:
        regional_button.icon = "times"
        regional_button.button_style = "warning"
        ranking_status.value = f"<span style='color: #b71c1c;'>Regional refresh failed: {err}</span>"
    finally:
        regional_button.disabled = False


def plot_regional_ranking(week_label):
    if regional_cases_df.empty:
        print("Click 'Fetch regional data' to build the regional ranking chart.")
        return
    if not week_label:
        print("Select a week.")
        return

    week = pd.to_datetime(week_label, errors="coerce")
    if pd.isna(week):
        print("Invalid week selected.")
        return
    if week not in regional_cases_df.index:
        print("Selected week not available in regional dataset.")
        return

    row = regional_cases_df.loc[week].dropna().sort_values(ascending=True)
    if row.empty:
        print("No regional values available for this week.")
        return

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(row.index, row.values, color="#72b7b2")
    max_value = row.max()
    for bar, value in zip(bars, row.values):
        ax.text(value + max(0.2, max_value * 0.01), bar.get_y() + bar.get_height() / 2, f"{value:.0f}", va="center")
    ax.set_title(f"Regional measles case ranking - week beginning {week:%Y-%m-%d}")
    ax.set_xlabel("Cases")
    ax.set_ylabel("Area")
    plt.tight_layout()
    plt.show()


available_metrics = list(df.columns)
default_selection = tuple(available_metrics[: min(3, len(available_metrics))])

series = wdg.SelectMultiple(
    options=available_metrics,
    value=default_selection,
    rows=max(3, min(8, len(available_metrics))),
    description="Metrics:",
    disabled=False,
)

scale = wdg.RadioButtons(
    options=["linear", "log"],
    description="Scale:",
    disabled=False,
)

weeks = wdg.IntSlider(
    value=min(52, len(df)),
    min=4,
    max=max(4, len(df)),
    step=1,
    description="Weeks:",
    continuous_update=False,
)

rolling_window = wdg.IntSlider(
    value=4,
    min=2,
    max=8,
    step=1,
    description="Rolling:",
    continuous_update=False,
)

controls = wdg.HBox([series, scale, weeks, rolling_window])

change_metric = wdg.Dropdown(
    options=available_metrics,
    value=available_metrics[0] if available_metrics else None,
    description="Metric:",
    disabled=False,
)

change_weeks = wdg.IntSlider(
    value=min(52, max(4, len(df) - 1)),
    min=4,
    max=max(4, len(df) - 1),
    step=1,
    description="Weeks:",
    continuous_update=False,
)

change_controls = wdg.HBox([change_metric, change_weeks])

regional_button = wdg.Button(
    description="Fetch regional data",
    disabled=False,
    button_style="",
    tooltip="Download all regional series for ranking",
    icon="download",
)
regional_button.on_click(fetch_regional_ranking_data)

ranking_week = wdg.Dropdown(
    options=[],
    value=None,
    description="Week:",
    disabled=True,
)

ranking_status = wdg.HTML(value="")
regional_controls = wdg.HBox([regional_button, ranking_week])


def refresh_graph():
    """Force a redraw of the main weekly chart after data refresh."""
    options = tuple(series.options)
    if len(options) >= 2:
        current = tuple(series.value)
        alternate = (options[0],) if current != (options[0],) else (options[1],)
        series.value = alternate
        series.value = current
        return

    current_weeks = weeks.value
    if weeks.max > 4:
        weeks.value = max(4, current_weeks - 1)
        weeks.value = current_weeks
        return

    current_scale = scale.value
    scale.value = "log" if current_scale == "linear" else "linear"
    scale.value = current_scale


def refresh_weekly_change():
    """Force a redraw of the week-over-week chart after data refresh."""
    if change_metric.value is None:
        return

    metric_options = list(change_metric.options)
    if len(metric_options) >= 2:
        current_metric = change_metric.value
        alternate_metric = metric_options[0] if current_metric != metric_options[0] else metric_options[1]
        change_metric.value = alternate_metric
        change_metric.value = current_metric
        return

    current_weeks = change_weeks.value
    if change_weeks.max > 4:
        change_weeks.value = max(4, current_weeks - 1)
        change_weeks.value = current_weeks


graph = wdg.interactive_output(
    plot_timeseries,
    {"gcols": series, "gscale": scale, "nweeks": weeks, "rolling_window": rolling_window},
)

weekly_change_graph = wdg.interactive_output(
    plot_weekly_change,
    {"metric": change_metric, "nweeks": change_weeks},
)

regional_ranking_graph = wdg.interactive_output(
    plot_regional_ranking,
    {"week_label": ranking_week},
)

display(wdg.HTML("<h4>1. Weekly bars with 4-week rolling average</h4>"), controls, graph)
display(wdg.HTML("<h4>2. Week-over-week change</h4>"), change_controls, weekly_change_graph)
display(wdg.HTML("<h4>3. Age distribution</h4>"))
plot_age_distribution()
display(wdg.HTML("<h4>4. Regional ranking by week</h4>"), regional_controls, ranking_status, regional_ranking_graph)


## Deploying the dashboard

Once your code is ready and you are satisfied with the appearance of the graphs, replace all the text boxes above with the explanations you would like a dashboard user to see. The next step is deploying the dashboard online - there are several [options](https://voila.readthedocs.io/en/stable/deploy.html) for this, we suggest deploying as a [Binder](https://mybinder.org/). This is basically the same technique that has been used to package this tutorial and to deploy this template dashboard. The instructions may seem a bit involved, but the actual steps are surprisingly easy - we will be going through them together during a live session. You will need an account on [GitHub](https://github.com/) for this - if you don't have one already, now it's the time to create it. 

**Author and License** Remember that if you deploy your dashboard as a Binder it will be publicly accessible. Change the copyright notice and take credit for your work. Also acknowledge your sources and license conditions by including a notice such as: "Based on UK Government [data](https://ukhsa-dashboard.data.gov.uk/) published by the [UK Health Security Agency](https://www.gov.uk/government/organisations/uk-health-security-agency), adapted from the [DIY Disease Tracking Dashboard Kit](https://github.com/fsmeraldi/diy-covid19dash). Released under the [GNU GPLv3.0 or later](https://www.gnu.org/licenses/)."